In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

ROOT_DIR = '/content/drive/MyDrive/Project Work DL'
DATA_DIR = os.path.join(ROOT_DIR, 'data')
TRAIN_DIR = os.path.join(DATA_DIR,'esercizi','train')
VALIDATION_DIR = os.path.join(DATA_DIR,'esercizi','validation')
TEST_DIR = os.path.join(DATA_DIR,'esercizi','test')

print(ROOT_DIR)
print(DATA_DIR)
print(TRAIN_DIR)



/content/drive/MyDrive/Project Work DL
/content/drive/MyDrive/Project Work DL/data
/content/drive/MyDrive/Project Work DL/data/esercizi/train


In [9]:

import os
import glob
import re

def getTextAndLabelsFromDataset(dir: str):
    texts, labels = [], []
    sub_folders = os.listdir(dir)

    for label in sub_folders:
        label_dir = os.path.join(dir,label)
        for text in glob.glob(os.path.join(label_dir, "*.txt")):
            with open(text, "r", encoding="utf-8") as f:
                text = f.read()
            if text:
                texts.append(text)
                labels.append(label)

    return texts, labels

In [27]:

import os
import glob
import re

def getTextAndLabelsFromDataset(dir: str):
    texts, labels = [], []
    sub_folders = os.listdir(dir)

    for label in sub_folders:
        label_dir = os.path.join(dir,label)
        for text in glob.glob(os.path.join(label_dir, "*.txt")):
            with open(text, "r", encoding="utf-8") as f:
                text = f.read()
                #rimozione cifra numero in numero
                text = re.sub(r'\d+([.,]\d+)?', 'numero', text)
                # Normalizza apostrofi e virgolette tipografiche in ASCII
                text = text.replace("’", "'").replace("‘", "'")
                text = text.replace("“", '"').replace("”", '"')
                text = re.sub(r'\s+([,.;:!?)])', r'\1', text)
                # Collassa spazi multipli
                text = re.sub(r'\s+', ' ', text).strip()
            if text:
                texts.append(text)
                labels.append(label)

    return texts, labels

In [28]:
from datasets import Dataset

train_text, train_labels = getTextAndLabelsFromDataset(TRAIN_DIR)
val_text, val_labels = getTextAndLabelsFromDataset(VALIDATION_DIR)
print(len(train_text))
print(len(val_text))

#converto le label in numeri per il dizionario di hf
classi_uniche = sorted(list(set(train_labels)))
label_to_id = {label: indice for indice, label in enumerate(classi_uniche)}

train_labels_id = [label_to_id[label] for label in train_labels]
val_labels_id = [label_to_id[label] for label in val_labels]

#utilizzo huggingface per caricare il dataset, invece di dataloader con pytorch (più semplice)
#hugging face utilizza i dizionari
hf_train_dict = Dataset.from_dict({"text":train_text, "label": train_labels_id})
hf_val_dict = Dataset.from_dict({"text":val_text, "label": val_labels_id})

483
70


In [29]:

print(train_text[0])

Chiara è molto impegnata perché stasera deve ricevere degli amici a cena. Deve ancora lavorare al pc per numero minuti (poi, la stampante lavorerà da sola per altri numero), preparare la lavastoviglie (ci metterà un quarto d'ora, poi la macchina andrà da sola per un'ora) e fare la doccia in numero minuti. Ma, prima della doccia, deve lavare per terra: bisogna preventivare altri numero minuti. Quanto minuti le servono come minimo?


In [33]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, TrainerCallback
import torch


# Custom Callback to log learning rate
class CustomLearningRateLogger(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        if state.global_step > 0 and 'optimizer' in kwargs:
            optimizer = kwargs['optimizer']
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {int(state.epoch)} - Current Learning Rate: {current_lr:.8f}")


#model_name = "mascIT/bert-tiny-ita"
#model_name = "dbmdz/bert-base-italian-cased"
model_name = "dbmdz/bert-base-italian-xxl-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # 'examples' è un dizionario che contiene una lista di testi
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=100
    )

train_tokenized = hf_train_dict.map(tokenize_function, batched=True)
val_tokenized = hf_val_dict.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = 7
)

training_param = TrainingArguments(
    eval_strategy="epoch",
    logging_strategy="epoch", # Added to log training loss at each epoch
    save_strategy="epoch",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=1e-5, #da bert tiny-ita, di default usa AdamW
    weight_decay=1e-4, #sempre da bert tiny-ita
    num_train_epochs=20,
    lr_scheduler_type = 'reduce_lr_on_plateau',
    metric_for_best_model = "eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=training_param.learning_rate, weight_decay=training_param.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

trainer = Trainer(
    model=model,
    args=training_param,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    optimizers=(optimizer, scheduler),
)

# Add the custom callback to the trainer
trainer.add_callback(CustomLearningRateLogger())

print("Inizio addestramento: \n")
trainer.train()


Map:   0%|          | 0/483 [00:00<?, ? examples/s]

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-italian-xxl-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Inizio addestramento: 



...

Epoch 19 - Current Learning Rate: 0.00000250


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 20 - Current Learning Rate: 0.00000250


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=320, training_loss=0.6782865475863218, metrics={'train_runtime': 1098.2347, 'train_samples_per_second': 8.796, 'train_steps_per_second': 0.291, 'total_flos': 496438847100000.0, 'train_loss': 0.6782865475863218, 'epoch': 20.0})

In [ ]:
from datetime import datetime, timezone, timedelta
import locale

# Imposta la localizzazione italiana per la formattazione
try:
    locale.setlocale(locale.LC_TIME, "it_IT.UTF-8")
except locale.Error:
    print("⚠️ Locale italiano non disponibile sul sistema. Verrà usato il formato predefinito.")

# Ottieni data e ora correnti in Italia (fuso orario UTC+1 o UTC+2 con ora legale)
italy_tz = timezone(timedelta(hours=2))  # UTC+2 per esempio (ora legale)
now_italy = datetime.now(italy_tz)

# Timestamp UNIX (secondi dal 1° gennaio 1970 UTC)
timestamp_unix = int(now_italy.timestamp())

# Formattazione in stile italiano
data_formattata = now_italy.strftime("%d-%b-%H:%M:%S")
print(data_formattata)

# Define a directory to save the model
folder_name = f"trained_model_{data_formattata}"
output_dir = os.path.join(ROOT_DIR, folder_name)

# Create the directory if it doesn't exist
#if not os.path.exists(output_dir):
#    os.makedirs(output_dir)

# Save the model and tokenizer
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model and tokenizer saved to {output_dir}")


⚠️ Locale italiano non disponibile sul sistema. Verrà usato il formato predefinito.
06-Jul-17:00:27


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [34]:
import os

temp = '/content/trainer_output'

!rm -rf {temp}